# 🎯 Module 09: Advanced RAG and Evaluation

> **Course:** Generative AI — From Scratch to GenAI Engineer  
> **Stage:** Stage 2 — Retrieval and RAG Foundations  
> **Module:** 09 — Advanced RAG and Evaluation  
> **Planned Date:** July 25, 2026  
> **Level:** Intermediate → Advanced  
> **Prerequisites:** Embeddings, Vector DBs, RAG Fundamentals  
> **Previous:** 08 — RAG Fundamentals  
> **Next:** 10 — LLM Frameworks and LangChain

---

Basic RAG works — but it **fails in predictable ways**. This module is about **finding, measuring, and fixing those failures**.

We'll cover:

- **Why basic RAG is not enough**
- The **RAG failure pipeline** (7 stages)
- **Advanced chunking** — fixed, overlap, structure-aware, parent-child
- **Query transformation** — rewriting, multi-query, decomposition
- **Hybrid search** — keyword + vector + RRF fusion
- **Reranking** — bi-encoders vs cross-encoders
- **Context optimization**
- **RAG evaluation** — retrieval metrics, generation metrics, system metrics
- Building an **evaluation dataset**
- Running **retrieval experiments**
- **Reliability & security**
- A **practical lab**: Advanced RAG Evaluation System

## 📚 Learning Objectives

By the end of this module you should be able to:

1. Identify common **failure modes** in basic RAG
2. Improve **chunking** and **metadata design**
3. Apply **query rewriting, decomposition, multi-query retrieval**
4. Combine keyword + vector search into **hybrid retrieval**
5. Use **reranking** to improve ordering
6. Understand **parent-child** & **contextual retrieval**
7. Apply **similarity thresholds & metadata filters** carefully
8. Evaluate **retrieval** vs **answer quality** separately
9. Compute **Recall@K, Precision@K, MRR, nDCG**
10. Build an **evaluation dataset**
11. Diagnose **hallucinations, missing evidence, irrelevant context**
12. Compare **alternative RAG configurations** through experiments

---
## ⚙️ 0. Setup & Imports

This notebook works **without an API key** — all metric math is local, and each RAG component is implemented from scratch for clarity.

In [1]:
import os
import json
import math
import hashlib
import numpy as np
from pathlib import Path
from collections import defaultdict

np.random.seed(42)
print('✅ Setup ready.')

HAS_KEY = bool(os.getenv('OPENAI_API_KEY'))
print(f'OPENAI_API_KEY present: {HAS_KEY}')

✅ Setup ready.
OPENAI_API_KEY present: False


---
## 🧠 1. Why Basic RAG Is Not Enough

Basic RAG:

1. Embed the user's question
2. Retrieve the nearest chunks
3. Add chunks to a prompt
4. Ask the LLM to answer

It **fails in predictable ways**.

| Problem | Example | Improvement |
| --- | --- | --- |
| Poor chunk boundaries | Definition split from explanation | Better chunking |
| Keyword mismatch | "refund" vs "reimbursement" | Hybrid search / query rewrite |
| Irrelevant top results | Similar wording, wrong subject | Reranking & filters |
| Missing information | Answer needs 2 passages | Decomposition / broader retrieval |
| Too much context | Evidence buried in noise | Filtering & compression |
| Unsupported answer | Model invents details | Grounding + evaluation |
| Stale documents | Old policies remain searchable | Versioning & re-index |

> 🎯 **Don't change the LLM first.** Identify whether the failure is in **ingestion, retrieval, context, or generation**.

---
## 🔬 2. The RAG Failure Pipeline

```text
Source Documents
       │
       ▼
[1] Ingestion and Parsing
       │
       ▼
[2] Chunking and Metadata
       │
       ▼
[3] Embedding and Indexing
       │
       ▼
[4] Retrieval and Ranking
       │
       ▼
[5] Context Construction
       │
       ▼
[6] LLM Generation
       │
       ▼
[7] Answer and Citation Validation
```

### Diagnose each stage

- **Ingestion** — Was the relevant info **extracted**?
- **Chunking** — Is required info **together** in useful chunks?
- **Indexing** — Were vectors created with the intended model?
- **Retrieval** — Did the system **find** the relevant evidence?
- **Context** — Did the prompt **include** important passages?
- **Generation** — Did the model **interpret** the evidence correctly?
- **Validation** — Do answer + citations **support** the claims?

> **Stage-by-stage debugging** prevents unnecessary model/infrastructure changes.

---
## 🧩 3. Advanced Chunking Strategies

> There is **no universally optimal chunk size**. It depends on the **document structure, query type, embedding model, and user needs**.

### 3.1 Fixed-Size
Split by **character or token** limit.

- ✅ Simple, predictable
- ❌ May split sentences / sections

### 3.2 Overlapping
Adjacent chunks share text.

```text
Chunk 1: [A B C D E]
Chunk 2:       [D E F G H]
Chunk 3:             [G H I J K]
```

- ✅ Preserves boundary context
- ❌ Duplicates content → higher storage & cost

### 3.3 Structure-Aware
Split by **headings, paragraphs, sections, tables, code blocks, pages**.

- ✅ Great for manuals, policies, docs
- ❌ Requires reliable parsing

### 3.4 Parent-Child Retrieval

```text
Parent Section
├── Child Chunk 1  ← retrieve this
├── Child Chunk 2
└── Child Chunk 3
```

Retrieve a **precise child**; return its **parent** to the LLM.

### 3.5 Comparison

| Strategy | Best for | Trade-off |
| --- | --- | --- |
| Fixed-size | General text | May split sections |
| Overlapping | Boundary-sensitive | Duplicates |
| Structure-aware | Manuals, policies | Parsing required |
| Parent-child | Long docs | Complex storage |
| Semantic | Changing topics | Higher cost |

In [2]:
# Implement three chunking strategies
def chunk_fixed(text, chunk_size=300):
    text = text.strip()
    return [text[i:i + chunk_size].strip() for i in range(0, len(text), chunk_size) if text[i:i + chunk_size].strip()]

def chunk_overlapping(text, chunk_size=300, overlap=60):
    text = text.strip()
    if overlap >= chunk_size:
        raise ValueError('overlap must be < chunk_size')
    out, start = [], 0
    while start < len(text):
        end = min(start + chunk_size, len(text))
        chunk = text[start:end].strip()
        if chunk:
            out.append(chunk)
        if end == len(text):
            break
        start = end - overlap
    return out

def chunk_structure_aware(text, sep='\n\n'):
    return [p.strip() for p in text.split(sep) if p.strip()]

# Sample text
sample = '''Annual Leave Policy

Employees may request annual leave through the approved process.
Requests must be reviewed by a manager.

Sick Leave Policy

Employees unable to work due to illness must notify their manager
and follow medical documentation procedures.

Attendance Policy

Employees must record their attendance daily.
'''

print('--- Fixed (size=120) ---')
for i, c in enumerate(chunk_fixed(sample, 120), 1):
    print(f'  {i}: {c[:60]!r}… ({len(c)} chars)')

print('\n--- Overlapping (size=120, overlap=30) ---')
for i, c in enumerate(chunk_overlapping(sample, 120, 30), 1):
    print(f'  {i}: {c[:60]!r}… ({len(c)} chars)')

print('\n--- Structure-aware (by blank line) ---')
for i, c in enumerate(chunk_structure_aware(sample), 1):
    print(f'  {i}: {c[:60]!r}… ({len(c)} chars)')

--- Fixed (size=120) ---
  1: 'Annual Leave Policy\n\nEmployees may request annual leave thro'… (120 chars)
  2: 'ager.\n\nSick Leave Policy\n\nEmployees unable to work due to il'… (120 chars)
  3: 'tion procedures.\n\nAttendance Policy\n\nEmployees must record t'… (82 chars)

--- Overlapping (size=120, overlap=30) ---
  1: 'Annual Leave Policy\n\nEmployees may request annual leave thro'… (120 chars)
  2: 'ests must be reviewed by a manager.\n\nSick Leave Policy\n\nEmpl'… (120 chars)
  3: 'lness must notify their manager\nand follow medical documenta'… (120 chars)
  4: 'olicy\n\nEmployees must record their attendance daily.'… (52 chars)

--- Structure-aware (by blank line) ---
  1: 'Annual Leave Policy'… (19 chars)
  2: 'Employees may request annual leave through the approved proc'… (104 chars)
  3: 'Sick Leave Policy'… (17 chars)
  4: 'Employees unable to work due to illness must notify their ma'… (110 chars)
  5: 'Attendance Policy'… (17 chars)
  6: 'Employees must record their atten

---
## 🔎 4. Query Transformation

The **user's original question** is not always the best search query.

### 4.1 Query Rewriting

**Original:** *"What about leave if I'm ill?"*  
**Rewritten:** *"What is the employee sick-leave policy, including notification and documentation requirements?"*

> ⚠️ The rewriting model must **not invent facts** or change what the user is asking.

### 4.2 Multi-Query Retrieval

Generate **multiple queries** from one question:

```text
Original: "How do I recover my account?"

Search queries:
1. How to reset a forgotten password
2. Account recovery procedure
3. Troubleshooting login access
```

Combine candidates & deduplicate.

### 4.3 Query Decomposition

Break a **complex question** into smaller ones:

```text
"Compare sick-leave and annual-leave policies, including eligibility."

→ 1. Sick-leave requirements?
→ 2. Annual-leave requirements?
→ 3. Eligibility differences?
→ 4. Approval procedures?
```

### 4.4 When to Use Each

| Technique | Useful when |
| --- | --- |
| Rewriting | Vague / unfamiliar wording |
| Multi-query | Docs use different terminology |
| Decomposition | Several independent info needs |
| Metadata filter | Known attributes restrict docs |

> ⚠️ Don't apply query transformation to **every** question automatically — measure latency & cost.

---
## 🔀 5. Hybrid Search

**Keyword search** is great for exact terms, IDs, error codes.  
**Vector search** is great for synonyms, paraphrase, meaning.

**Hybrid search** combines them.

### Architecture

```text
                    User Query
                        │
             ┌──────────┴──────────┐
             ▼                     ▼
       Keyword Search         Vector Search
             │                     │
             ▼                     ▼
       Ranked Results         Ranked Results
             │                     │
             └──────────┬──────────┘
                        ▼
                 Result Fusion
                        │
                        ▼
                    Reranking
                        │
                        ▼
                 Final Documents
```

### Reciprocal Rank Fusion (RRF)

$$\operatorname{RRF}(d) = \sum_{r \in R} \frac{1}{k_0 + \operatorname{rank}_r(d)}$$

- `d` — candidate document
- `R` — set of ranked lists
- `rank_r(d)` — rank of `d` in list `r`
- `k_0` — smoothing constant (often 60)

A document that appears **high in multiple lists** gets a strong combined score.

In [3]:
# Tiny hybrid search demo — keyword (BM25-like) + vector + RRF
def tokenize(text):
    return text.lower().split()

def keyword_score(query, doc):
    q_tokens = set(tokenize(query))
    d_tokens = tokenize(doc)
    return sum(1 for t in d_tokens if t in q_tokens)

def pseudo_embed(text, dim=64):
    h = hashlib.sha256(text.encode()).digest()
    v = np.frombuffer(h, dtype=np.uint8).astype(float)
    v = np.resize(v, dim)
    return v / (np.linalg.norm(v) + 1e-9)

def vector_score(query, doc):
    q, d = pseudo_embed(query), pseudo_embed(doc)
    return float(np.dot(q, d))

def rrf_fuse(ranked_lists, k0=60):
    scores = defaultdict(float)
    for ranked in ranked_lists:
        for rank, doc_id in enumerate(ranked, start=1):
            scores[doc_id] += 1.0 / (k0 + rank)
    return sorted(scores.items(), key=lambda x: -x[1])

# Sample corpus
docs = {
    'd1': 'Reset your password from account settings',
    'd2': 'Billing portal to update payment information',
    'd3': 'Server troubleshooting for engineers',
    'd4': 'Forgot login credentials — recover access',
}

query = 'forgot my login password'

# Keyword-ranked list
kw = sorted(docs, key=lambda i: -keyword_score(query, docs[i]))
print('Keyword ranking:', kw)

# Vector-ranked list
vec = sorted(docs, key=lambda i: -vector_score(query, docs[i]))
print('Vector  ranking:', vec)

# RRF fusion
fused = rrf_fuse([kw, vec])
print('\nRRF fused ranking:')
for doc_id, score in fused:
    print(f'  {doc_id}  score={score:.4f}  {docs[doc_id]}')

Keyword ranking: ['d4', 'd1', 'd2', 'd3']
Vector  ranking: ['d2', 'd4', 'd3', 'd1']

RRF fused ranking:
  d4  score=0.0325  Forgot login credentials — recover access
  d2  score=0.0323  Billing portal to update payment information
  d1  score=0.0318  Reset your password from account settings
  d3  score=0.0315  Server troubleshooting for engineers


---
## 🎯 6. Reranking

First-stage retrieval is optimized for **speed**.  
A **reranker** then scores candidates **more carefully**.

```text
Query
  │
  ▼
Retrieve Top 30 Candidates
  │
  ▼
Reranker Scores Query + Candidate
  │
  ▼
Select Top 5
  │
  ▼
Build LLM Context
```

### Bi-Encoder vs Cross-Encoder

| Feature | Bi-Encoder | Cross-Encoder |
| --- | --- | --- |
| Processing | Query & doc separately | Query + doc together |
| Speed | Fast, scalable | Slow, accurate |
| Use | 1st-stage retrieval | 2nd-stage reranking |

> ⚠️ A cross-encoder score is **not** a calibrated probability.

### When to add a reranker

- Relevant docs appear in candidates but **rank too low**
- Top results are **related but not useful**
- Better context is **worth the extra latency**

> ⚠️ A reranker **cannot recover** documents the first stage never found.

---
## 📦 7. Context Optimization

Retrieval is only half of the problem — you must decide **what to include** in the prompt.

### Techniques

- **Top-K tuning** — test how many chunks give enough evidence
- **Context deduplication** — remove duplicate passages
- **Relevance filtering** — drop candidates below an evaluated threshold
- **Parent-child expansion** — add surrounding context when needed
- **Context compression** — retain useful info while reducing length
- **Source-aware ordering** — preserve source relationships & metadata

### 🧭 Important principle

**Never choose `top_k` or a similarity threshold by guessing.**

1. Build representative evaluation queries
2. Inspect retrieved evidence
3. Measure retrieval & answer quality
4. Compare latency & token usage
5. Pick a config that meets requirements

> Very high threshold → may remove useful evidence.  
> Very low threshold → may add irrelevant context.

---
## 📊 8. RAG Evaluation

Evaluate **retrieval, generation, citations, cost, and latency** independently.

### 8.1 Retrieval metrics

#### Precision@K
$$\text{Precision@K} = \frac{\text{Relevant results in top-K}}{K}$$

#### Recall@K
$$\text{Recall@K} = \frac{\text{Relevant results in top-K}}{\text{Total relevant results}}$$

#### Mean Reciprocal Rank (MRR)
$$\text{MRR} = \frac{1}{|Q|} \sum_{i=1}^{|Q|} \frac{1}{\text{rank}_i}$$

Queries with no relevant results contribute **0**.

#### nDCG
**Normalized Discounted Cumulative Gain** handles **graded relevance** — rewards systems that put highly-relevant docs near the top.

### 8.2 Generation metrics

| Metric | Question |
| --- | --- |
| Correctness | Is the answer factually correct? |
| Faithfulness / Groundedness | Claims supported by evidence? |
| Answer relevance | Does it address the question? |
| Citation correctness | Do cited sources support claims? |
| Completeness | Includes required info? |
| Abstention quality | Admits when evidence is insufficient? |

### 8.3 System metrics

- End-to-end latency
- Retrieval latency
- Generation latency
- Token usage (input + output)
- Embedding + reranking cost
- Error rate
- Cache hit rate
- Indexing duration

---
## 🧪 9. Build an Evaluation Dataset

Create a dataset of **realistic questions** before making major changes.

### Example `evaluation/questions.json`

```json
[
  {
    "id": "q001",
    "question": "How can an employee request annual leave?",
    "relevant_sources": ["annual_leave_policy"],
    "expected_answer_points": [
      "Use the approved leave-management process"
    ]
  },
  {
    "id": "q002",
    "question": "What should an employee do when sick?",
    "relevant_sources": ["sick_leave_policy"],
    "expected_answer_points": [
      "Notify the manager",
      "Follow the sick-leave procedure"
    ]
  },
  {
    "id": "q003",
    "question": "What is the company's office address?",
    "relevant_sources": [],
    "expected_answer_points": [
      "The available documents do not provide this information"
    ]
  }
]
```

### Evaluation workflow

1. Create representative questions
2. Label relevant documents / chunks
3. Record expected answer points
4. Run every query through the **same** pipeline
5. Calculate retrieval metrics
6. Review generated answers & citations
7. Compare configs on the **same** dataset
8. Investigate failures & repeat

> Keep a **separate test set** so you don't tune every decision against the same examples.

In [4]:
# Retrieval metric implementations
def precision_at_k(retrieved, relevant, k):
    top = retrieved[:k]
    if k == 0:
        return 0.0
    return sum(1 for r in top if r in relevant) / k

def recall_at_k(retrieved, relevant, k):
    if not relevant:
        return 0.0
    top = retrieved[:k]
    return sum(1 for r in top if r in relevant) / len(relevant)

def reciprocal_rank(retrieved, relevant):
    for i, r in enumerate(retrieved, start=1):
        if r in relevant:
            return 1.0 / i
    return 0.0

def mrr(all_retrieved, all_relevant):
    if not all_retrieved:
        return 0.0
    return sum(reciprocal_rank(r, rel) for r, rel in zip(all_retrieved, all_relevant)) / len(all_retrieved)

def dcg(relevances):
    return sum(rel / math.log2(i + 1) for i, rel in enumerate(relevances, start=1))

def ndcg_at_k(retrieved, relevance_map, k):
    gains  = [relevance_map.get(doc, 0.0) for doc in retrieved[:k]]
    ideal  = sorted(relevance_map.values(), reverse=True)[:k]
    idcg = dcg(ideal)
    return (dcg(gains) / idcg) if idcg > 0 else 0.0

# Demo
retrieved = ['d1', 'd4', 'd2', 'd3']
relevant  = {'d1', 'd4'}
relevance_map = {'d1': 3, 'd4': 2, 'd2': 1, 'd3': 0}

print(f'Precision@2  = {precision_at_k(retrieved, relevant, 2):.3f}')
print(f'Precision@3  = {precision_at_k(retrieved, relevant, 3):.3f}')
print(f'Recall@3     = {recall_at_k(retrieved, relevant, 3):.3f}')
print(f'RR           = {reciprocal_rank(retrieved, relevant):.3f}')
print(f'nDCG@4        = {ndcg_at_k(retrieved, relevance_map, 4):.3f}')

Precision@2  = 1.000
Precision@3  = 0.667
Recall@3     = 1.000
RR           = 1.000
nDCG@4        = 1.000


---
## 🛠️ 10. Practical Implementation — Retrieval Experiment

Compare **multiple retrieval configurations** on the same dataset.

| Config | Retrieval Method | Reranking |
| --- | --- | --- |
| A | Vector | No |
| B | Keyword | No |
| C | Hybrid | No |
| D | Hybrid | Yes |

Record results as a table:

| Config | Recall@5 | MRR | Groundedness | Latency | Cost |
| --- | ---: | ---: | ---: | ---: | ---: |
| A | real | real | real | real | real |
| B | real | real | real | real | real |
| C | real | real | real | real | real |
| D | real | real | real | real | real |

> ⚠️ **Do NOT invent numbers.** Run the experiment on your dataset.

In [5]:
# Reproducible experiment scaffold using the demo corpus
import time

# Docs (with topic labels)
exp_docs = {
    'd1': ('Password reset steps', 'support'),
    'd2': ('Billing portal guide', 'billing'),
    'd3': ('Server troubleshooting', 'engineering'),
    'd4': ('Login recovery procedure', 'support'),
    'd5': ('Refund policy details', 'billing'),
}

def make_vector(text, dim=64):
    h = hashlib.sha256(text.encode()).digest()
    v = np.frombuffer(h, dtype=np.uint8).astype(float)
    v = np.resize(v, dim)
    return v / (np.linalg.norm(v) + 1e-9)

exp_vectors = {i: make_vector(t) for i, (t, _) in exp_docs.items()}

def vec_rank(q, topk=3):
    qv = make_vector(q)
    scored = sorted(exp_vectors.items(), key=lambda kv: -float(np.dot(qv, kv[1])))
    return [doc_id for doc_id, _ in scored[:topk]]

def kw_rank(q, topk=3):
    qt = set(q.lower().split())
    scored = sorted(exp_docs.items(), key=lambda kv: -sum(1 for t in kv[1][0].lower().split() if t in qt))
    return [doc_id for doc_id, _ in scored[:topk]]

def hybrid_rank(q, topk=3, k0=60):
    kw  = kw_rank(q, topk=len(exp_docs))
    vec = vec_rank(q, topk=len(exp_docs))
    scores = defaultdict(float)
    for ranked in (kw, vec):
        for i, doc_id in enumerate(ranked, start=1):
            scores[doc_id] += 1.0 / (k0 + i)
    return [d for d, _ in sorted(scores.items(), key=lambda x: -x[1])[:topk]]

# Evaluation query
q = 'forgot password login'
relevant = {'d1', 'd4'}

print(f'Query: {q!r}\nRelevant: {relevant}\n')
for name, fn in [('Vector', vec_rank), ('Keyword', kw_rank), ('Hybrid', hybrid_rank)]:
    t0 = time.time()
    ranked = fn(q, topk=3)
    dt = time.time() - t0
    print(f'{name:<8}  top3={ranked}  P@3={precision_at_k(ranked, relevant, 3):.3f}  RR={reciprocal_rank(ranked, relevant):.3f}  ({dt*1000:.1f} ms)')

Query: 'forgot password login'
Relevant: {'d4', 'd1'}

Vector    top3=['d2', 'd1', 'd3']  P@3=0.333  RR=0.500  (0.1 ms)
Keyword   top3=['d1', 'd4', 'd2']  P@3=0.667  RR=1.000  (0.0 ms)
Hybrid    top3=['d1', 'd2', 'd4']  P@3=0.667  RR=1.000  (0.1 ms)


---
## 🔐 11. Reliability & Security

Advanced retrieval introduces additional failure modes.

### Document Permissions
Apply access-control rules **before** supplying retrieved content to the model. Never let a user gain access to a doc just because it's **similar** to their query.

### Prompt Injection
Documents may contain instructions designed to influence the model. Treat retrieved content as **untrusted data**. Keep system instructions separate. Never let document text **authorize actions**.

### Sensitive Information
Avoid exposing secrets or unnecessary PII in prompts, logs, evaluation datasets, or debug output.

### Citation Reliability
Do not generate source references from guesswork. Preserve source IDs & locations. **Validate** citations against retrieved evidence.

### Evaluation Integrity
Don't rely exclusively on a single LLM judge or a few hand-picked queries. Combine automated metrics with **human-reviewed examples**.

### Change Management

Track changes to:

- Source documents
- Chunking rules
- Embedding models
- Retrieval settings
- Reranking models
- Prompts
- Evaluation datasets

This makes **regressions** easier to diagnose.

---
## 🏋️ 12. Practical Project — Advanced RAG Evaluation System

### 🎯 Goal
Upgrade your PDF RAG Assistant into a **measurable, comparable, improvable** retrieval pipeline.

### Core requirements

- [ ] Realistic document collection
- [ ] Preserve document & chunk metadata
- [ ] Baseline vector retrieval
- [ ] Keyword / hybrid retrieval
- [ ] ≥ 2 chunking strategies compared
- [ ] Reranking **or** documented justification for skipping
- [ ] Labeled evaluation dataset
- [ ] Compute **Recall@K, Precision@K, MRR**
- [ ] Evaluate answer correctness & groundedness
- [ ] Track citation quality
- [ ] Measure latency & cost
- [ ] Test questions with **insufficient evidence**
- [ ] Document security & access-control assumptions
- [ ] Produce an **experiment report** with real results

### Suggested structure

```text
09-Advanced-RAG-and-Evaluation/
├── README.md
├── requirements.txt
├── .env.example
├── src/
│   ├── ingestion/
│   │   ├── loaders.py
│   │   └── chunking.py
│   ├── retrieval/
│   │   ├── vector_search.py
│   │   ├── keyword_search.py
│   │   ├── hybrid_search.py
│   │   └── reranking.py
│   ├── generation/
│   │   └── answer.py
│   └── evaluation/
│       ├── retrieval_metrics.py
│       ├── answer_metrics.py
│       └── run_evaluation.py
├── data/
│   └── documents/
├── evaluation/
│   ├── questions.json
│   └── results/
├── tests/
│   ├── test_chunking.py
│   ├── test_retrieval.py
│   └── test_metrics.py
└── reports/
    └── experiment_report.md
```

### Stretch goals

1. Query rewriting & decomposition
2. Compare dense / sparse / hybrid
3. Parent-child retrieval
4. Multiple embedding models
5. Reranker vs baseline
6. FastAPI endpoint
7. Evaluation dashboard
8. Regression tests on retrieval changes

---
## 🎤 13. Interview Questions

**Q1. Why can basic RAG fail?**  
Evidence may not be extracted, chunked, indexed, or retrieved correctly. Even with good evidence, the LLM may misinterpret it.

**Q2. What is hybrid search?**  
Combines **keyword-based retrieval** with **vector-based retrieval** to use both exact matching and semantic similarity.

**Q3. What is reranking?**  
Takes an initial candidate set and **scores/reorders** it before context construction.

**Q4. What is query rewriting?**  
Transforms a user's question into a clearer or more retrieval-friendly query while preserving intent.

**Q5. What is query decomposition?**  
Breaks a complex question into smaller **information needs** retrieved and answered separately.

**Q6. What is parent-child retrieval?**  
Retrieves a focused **child chunk** but returns its larger **parent section** for context.

**Q7. Recall@K vs Precision@K?**  
Recall = how many known relevant items were retrieved. Precision = how many of the retrieved top-k are relevant.

**Q8. What does MRR measure?**  
Mean Reciprocal Rank — how highly the **first relevant result** appears across a set of queries.

**Q9. Why evaluate retrieval and generation separately?**  
A correct answer needs **both** relevant evidence **and** correct use of it. Separate measurements pinpoint the failing stage.

**Q10. Does reranking always improve RAG?**  
No. It adds latency & cost. It **cannot recover** docs missing from the initial candidate set.

**Q11. How do you evaluate groundedness?**  
Check whether claims are supported by retrieved evidence — via human review, deterministic checks, and validated automated eval.

**Q12. How should RAG handle missing evidence?**  
Avoid fabrication, communicate the limitation, optionally ask for clarification or route to another trusted source.

---
## ✅ 14. Module Completion Checklist

- [ ] Understand the major RAG failure modes
- [ ] Compare **fixed / overlapping / structure-aware** chunking
- [ ] Understand **parent-child** retrieval
- [ ] Implement **query rewriting / decomposition**
- [ ] Compare **keyword / vector / hybrid**
- [ ] Understand **Reciprocal Rank Fusion**
- [ ] Evaluate **reranking**
- [ ] Experiment with **context selection**
- [ ] Calculate **Precision@K, Recall@K, MRR**
- [ ] Evaluate **answer correctness, groundedness, citations**
- [ ] Measure **latency & cost**
- [ ] Complete the **Advanced RAG Evaluation System**
- [ ] Document **real experimental results** & limitations

---
## 🏁 15. Key Takeaways

- Advanced RAG is about improving the **whole pipeline** — not just the LLM
- **Chunking, query transformation, hybrid search, reranking** solve different problems
- **Retrieval metrics** and **answer metrics** measure different aspects
- **Evaluation datasets** make improvements **measurable & repeatable**
- A reranker **cannot retrieve** what first-stage search never found
- **More context ≠ better** — evidence quality matters
- **Security, traceability, latency, cost** must be evaluated alongside answer quality

### 🧭 Golden Rule

> **Debug the pipeline stage by stage. Measure retrieval and generation separately. Change one thing at a time and prove it helped.**

---

### 🔗 What's Next?

**Module 10 — LLM Frameworks and LangChain**

- How frameworks organize **model calls, prompts, retrieval, output parsing**
- Compare **framework-based vs direct SDK code**
- When abstractions **help** vs when they add **unnecessary complexity**